# Redrob Candidate Ranking — Sandbox Demo

This notebook reproduces the **ranking step** on a small candidate sample, CPU-only, in well under 5 minutes.

It clones the repo, installs deps, and runs `rank.py` on a 100-candidate sample to produce a ranked CSV.

The full 100K reproduction happens in the organizers' own Stage-3 sandbox; this is the small-sample sanity check required by submission_spec Section 10.5.

## 1. Clone the repo
Replace the URL with your repo if running a fork.

In [ ]:
!git clone https://github.com/USERNAME/REPO.git redrob
%cd redrob

## 2. Install dependencies (CPU wheels)
The ranking step needs only pandas, numpy, scipy, lightgbm, scikit-learn, pyarrow. No GPU, no network at run time.

In [ ]:
!pip install -q pandas numpy scipy scikit-learn lightgbm pyarrow

## 3. Build a 100-candidate sample
The repo ships `sample_candidates.json` (50 rows) and the precomputed `features_100k.parquet`.
We take 100 candidate IDs that exist in the feature table and write them as a small `candidates.jsonl`.
(In the organizers' Stage-3 run, the full released `candidates.jsonl` is used instead.)

In [ ]:
import json, pandas as pd
df = pd.read_parquet('artifacts/features_100k.parquet')
sample_ids = list(df.index[:100])
# Minimal candidate records are enough for rank.py (it reads features from the parquet;
# the candidate file is used for ID filtering and reasoning text). We reuse the full
# records from sample_candidates if present, else emit minimal stubs.
try:
    full = {c['candidate_id']: c for c in json.load(open('sample_candidates.json'))}
except Exception:
    full = {}
with open('sandbox_candidates.jsonl', 'w') as f:
    for cid in sample_ids:
        rec = full.get(cid, {'candidate_id': cid, 'profile': {}, 'career_history': [], 'skills': [], 'redrob_signals': {}})
        f.write(json.dumps(rec) + '\n')
print('wrote', len(sample_ids), 'candidates to sandbox_candidates.jsonl')

## 4. Run the ranking step (CPU, < 5 min)
This is the exact command the organizers reproduce.

In [ ]:
import time
t0 = time.time()
!python rank.py --candidates sandbox_candidates.jsonl --out sandbox_submission.csv
print(f'\nelapsed: {time.time()-t0:.1f}s')

## 5. Show the ranked output

In [ ]:
import pandas as pd
out = pd.read_csv('sandbox_submission.csv')
print('rows:', len(out), '| distinct reasoning:', out.reasoning.nunique())
out.head(15)